In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import matplotlib.pyplot as plt


### 1. Load DataSets

In [2]:
order_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_orders_dataset.csv")
customer_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_customers_dataset.csv")
payment_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_order_payments_dataset.csv")
product_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_products_dataset.csv")
review_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_order_reviews_dataset.csv")
geolocation_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_geolocation_dataset.csv")
seller_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_sellers_dataset.csv")
item_data = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/data/olist_order_items_dataset.csv")

### 2. Merging The Table

In [3]:
payment_agg = payment_data.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max',
    'payment_type': lambda x: x.mode()[0]
}).reset_index()

In [4]:
item_agg = item_data.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum',
    'order_item_id': 'count'
}).reset_index()

item_agg.rename(columns={
    'order_item_id': 'total_items'
}, inplace=True)

In [5]:
review_agg = review_data.groupby('order_id').agg({
    'review_score': 'mean'
}).reset_index()

In [6]:
geo = geolocation_data.groupby(
    'geolocation_zip_code_prefix'
).agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

In [7]:
df = (
    order_data
    .merge(customer_data, on='customer_id', how='left')
    .merge(payment_agg, on='order_id', how='left')
    .merge(item_agg, on='order_id', how='left')
    .merge(review_agg, on='order_id', how='left')
)

df = df.merge(
    geo,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

In [8]:
print(df.shape)

print(df['order_id'].nunique())

print(df.shape[0] / df['order_id'].nunique())

(99441, 24)
99441
1.0


In [9]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_type,price,freight_value,total_items,review_score,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,voucher,29.99,8.72,1.0,4.0,3149.0,-23.576983,-46.587161,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,boleto,118.70,22.76,1.0,4.0,47813.0,-12.177924,-44.660711,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,credit_card,159.90,19.22,1.0,5.0,75265.0,-16.745150,-48.514783,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,credit_card,45.00,27.20,1.0,5.0,59296.0,-5.774190,-35.271143,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,credit_card,19.90,8.72,1.0,5.0,9195.0,-23.676370,-46.514627,santo andre,SP


In [10]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_value', 'payment_installments',
       'payment_type', 'price', 'freight_value', 'total_items', 'review_score',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')

### Handling Missing Value

In [11]:
df.isnull().sum()[df.isnull().sum()>0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
payment_value                       1
payment_installments                1
payment_type                        1
price                             775
freight_value                     775
total_items                       775
review_score                      768
geolocation_zip_code_prefix       278
geolocation_lat                   278
geolocation_lng                   278
geolocation_city                  278
geolocation_state                 278
dtype: int64

In [12]:
# order_approved_at
df[df['order_approved_at'].isnull()]['order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [13]:
# order_delivered_customer_date
df[df['order_delivered_carrier_date'].isnull()]['order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [14]:
#creating a new feature
df['is_delivered'] = (
    df['order_delivered_customer_date'].notnull()
)

In [15]:
# Payment column
df['payment_value'] = df['payment_value'].fillna(
    df['payment_value'].median(),
)

df['payment_installments'] = df['payment_installments'].fillna(
    df['payment_installments'].median(),
)

df['payment_type'] = df['payment_type'].fillna(
    df['payment_type'].mode()[0],
)

In [16]:
# price / freight / total_items
df[df['price'].isnull()]['order_status'].value_counts()

df['price'] = df['price'].fillna(0)
df['freight_value'] = df['freight_value'].fillna(0)
df['total_items'] = df['total_items'].fillna(0)

In [17]:
# review_score

df['has_review'] = df['review_score'].notnull()

In [18]:
# Geolocation
df['geolocation_lat'] = df['geolocation_lat'].fillna(
    df['geolocation_lat'].median(),
)

df['geolocation_lng'] = df['geolocation_lng'].fillna(
    df['geolocation_lng'].median(),
)

# catorigcal column
df['geolocation_city'] = df['geolocation_city'].fillna(
    df['geolocation_city'].mode()[0],
)

df['geolocation_state'] = df['geolocation_state'].fillna(
    df['geolocation_state'].mode()[0],
)

In [19]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_value', 'payment_installments',
       'payment_type', 'price', 'freight_value', 'total_items', 'review_score',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state', 'is_delivered', 'has_review'],
      dtype='str')

### Data Summary

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99441 non-null  str    
 1   customer_id                    99441 non-null  str    
 2   order_status                   99441 non-null  str    
 3   order_purchase_timestamp       99441 non-null  str    
 4   order_approved_at              99281 non-null  str    
 5   order_delivered_carrier_date   97658 non-null  str    
 6   order_delivered_customer_date  96476 non-null  str    
 7   order_estimated_delivery_date  99441 non-null  str    
 8   customer_unique_id             99441 non-null  str    
 9   customer_zip_code_prefix       99441 non-null  int64  
 10  customer_city                  99441 non-null  str    
 11  customer_state                 99441 non-null  str    
 12  payment_value                  99441 non-null  float64
 1

In [21]:
# Convert  Date Columns
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])


In [22]:
df.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,payment_value,payment_installments,price,freight_value,total_items,review_score,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,99441,99281,97658,96476,99441,99441.000000,99441.000000,99441.000000,99441.000000,99441.000000,99441.000000,98673.000000,99163.000000,99441.000000,99441.000000
mean,2017-12-31 08:43:12.776581,2017-12-31 18:35:24.098800,2018-01-04 21:49:48.138278,2018-01-14 12:09:19.035542,2018-01-24 03:08:37.730111,35137.474583,160.989707,2.930512,136.680481,22.645685,1.132833,4.086793,35057.887176,-21.196071,-46.176714
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,1003.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1003.000000,-33.689948,-72.668881
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.250000,2017-09-25 22:07:22.250000,2017-10-03 00:00:00,11347.000000,62.010000,1.000000,45.000000,13.720000,1.000000,4.000000,11320.000000,-23.588380,-48.082298
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00,24416.000000,105.290000,2.000000,85.000000,17.090000,1.000000,5.000000,24350.000000,-22.924970,-46.630647
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.250000,2018-05-25 00:00:00,58900.000000,176.970000,4.000000,149.900000,23.920000,1.000000,5.000000,58406.500000,-20.140360,-43.607441
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,99990.000000,13664.080000,24.000000,13440.000000,1794.960000,21.000000,5.000000,99990.000000,42.184003,-8.723762
std,NaN,NaN,NaN,NaN,NaN,29797.938996,221.950211,2.715673,210.172081,21.659564,0.545666,1.346274,29784.314664,5.601539,4.050465


### Outlier Detection

In [23]:
q1 = df['price'].quantile(0.25)
q3 = df["price"].quantile(0.75)

IQR = q3 - q1

lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR

price_outliers = df[df['price'] > upper_bound]

print(f'statistical summary of Price:')
print(f"upper bound (max logical price): ${upper_bound:.2f}")
print(f"Number of Outliers: {len(price_outliers)} out of {len(df)}")
print(f"Percentage of Outliers: {(len(price_outliers)/len(df))*100:.2f}%")

statistical summary of Price:
upper bound (max logical price): $307.25
Number of Outliers: 7903 out of 99441
Percentage of Outliers: 7.95%


In [24]:
high_freight_orders = df[df['freight_value'] > df['price']]

print(f"--- Business Anomaly Report ---")
print(f"Orders with Shipping > Price: {len(high_freight_orders)}")
print(f"Max Freight Value found: ${df['freight_value'].max():.2f}")

--- Business Anomaly Report ---
Orders with Shipping > Price: 3159
Max Freight Value found: $1794.96


### Feature Engineering

In [25]:
# how many days it take to deliver the product
df["delivery_days"] = (df['order_delivered_customer_date'] - df["order_purchase_timestamp"]).dt.days

In [26]:
# approval delay in hours

df["approval_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600

In [27]:
# shipping delay
df['shipping_days'] = (df['order_delivered_carrier_date'] - df['order_purchase_timestamp']).dt.days

In [28]:
# Late Delivery
df['is_late'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
)

### Extract Time Features

In [29]:
df['purchase_year'] = (
    df['order_purchase_timestamp'].dt.year
)

df['purchase_month'] = (
    df['order_purchase_timestamp'].dt.month
)

df['purchase_day'] = (
    df['order_purchase_timestamp'].dt.day
)

df['purchase_hour'] = (
    df['order_purchase_timestamp'].dt.hour
)

df['purchase_day_name'] = (
    df['order_purchase_timestamp'].dt.day_name()
)

### Revenue Analysis

In [30]:
# total revenue
df["payment_value"].sum()


np.float64(16008977.41)

In [31]:
# monthely Revenue Trend
monthly_sales = df.groupby(
    df['order_purchase_timestamp'].dt.to_period('M')
)['payment_value'].sum().sort_values(ascending=False)

monthly_sales

order_purchase_timestamp
2017-11    1194882.80
2018-04    1160785.48
2018-03    1159652.12
2018-05    1153982.15
2018-01    1115004.18
2018-07    1066540.75
2018-06    1023880.50
2018-08    1022425.32
2018-02     992463.34
2017-12     878401.48
2017-10     779677.88
2017-09     727762.45
2017-08     674396.32
2017-05     592918.82
2017-07     592382.92
2017-06     511276.38
2017-03     449863.60
2017-04     417788.03
2017-02     291908.01
2017-01     138488.04
2016-10      59090.48
2018-09       4439.54
2018-10        589.67
2016-09        357.53
2016-12         19.62
Freq: M, Name: payment_value, dtype: float64

In [32]:
# Average order value
df['payment_value'].mean()

np.float64(160.98970655966855)

### Customer Analysis

In [56]:
# TOtal Customer

x = df['customer_unique_id'].count()
x

np.int64(99441)

In [34]:
# Revenue by State
df.groupby('customer_state')[
    'payment_value'
].sum().sort_values(ascending=False)

customer_state
SP    5998332.25
RJ    2144379.69
MG    1872257.26
RS     890898.54
PR     811156.38
SC     623086.43
BA     616645.82
DF     355141.08
GO     350092.31
ES     325967.55
PE     324850.44
CE     279464.03
PA     218295.85
MT     187029.29
MA     152523.02
PB     141545.72
MS     137534.84
PI     108523.97
RN     102718.13
AL      96962.06
SE      75246.25
TO      61485.33
RO      60866.20
AM      27966.93
AC      19680.62
AP      16262.80
RR      10064.62
Name: payment_value, dtype: float64

In [35]:
# TOp city
df["customer_city"].value_counts().head(10)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64

**SP completely dominates  sales, generating nearly triple the revenue of any other state. RJ and MG follow as next biggest hubs, making the Southeast region absolute core market. Meanwhile, states like AP and RR barely scratch the surface, pointing to clear growth opportunities or logistical friction in the North.**

### Logistics Analysis

In [36]:
# Average delivery Time
df['delivery_days'].mean()

np.float64(12.094085575687217)

In [37]:
# Late delivery %
df['is_late'].mean() * 100

np.float64(7.870998883760219)

In [38]:
# states with Slowest delivery
df.groupby('customer_state')[
    'delivery_days'
].mean().sort_values(ascending=False)

customer_state
RR    28.975610
AP    26.731343
AM    25.986207
AL    24.040302
PA    23.316068
MA    21.117155
SE    21.029851
CE    20.817826
AC    20.637500
PB    19.953578
PI    18.993697
RO    18.913580
BA    18.866400
RN    18.824895
PE    17.965474
MT    17.593679
TO    17.226277
ES    15.331830
MS    15.191155
GO    15.150741
RJ    14.849186
RS    14.819237
SC    14.479560
DF    12.509135
MG    11.543813
PR    11.526711
SP     8.298061
Name: delivery_days, dtype: float64

### Customer Satisfaction Analysis

In [39]:
#Average review score
df['review_score'].mean()

np.float64(4.0867934152875325)

In [40]:
#late delivery impact on reviews
df.groupby('is_late')[
    'review_score'
].mean()

is_late
False    4.214778
True     2.566562
Name: review_score, dtype: float64

### Payment Analysis

In [41]:
# most use payment type
df["payment_type"].value_counts()

payment_type
credit_card    76133
boleto         19784
voucher         1994
debit_card      1527
not_defined        3
Name: count, dtype: int64

In [42]:
# highest spending payment type

df.groupby('payment_type')[
    'payment_value'
].mean().sort_values(ascending=False)

payment_type
credit_card    166.564802
boleto         145.034435
debit_card     142.724158
voucher        120.661108
not_defined      0.000000
Name: payment_value, dtype: float64

In [43]:
# installments using pytment type
df.groupby('payment_type')[
    'payment_installments'
].count().sort_values(ascending=False)

payment_type
credit_card    76133
boleto         19784
voucher         1994
debit_card      1527
not_defined        3
Name: payment_installments, dtype: int64

**Here we can see the Credit card Dominate all the payment type.driving over 76% of all purchases because they allow customers to split payments into installments.Boleto solidly holds second place as the go-to choice for cash buyers, while vouchers and debit cards represent tiny niche markets [1]. This heavy reliance on credit cards means optimizing checkout speeds and fraud protection for card users will directly protect your core revenue.**

### Visulization

In [44]:
import plotly.express as px


df_grouped = df.groupby('customer_state', as_index=False)['payment_value'].sum()

fig = px.bar(
    df_grouped, 
    x='customer_state', 
    y='payment_value',
    title='Total Payment Value by State',
    text_auto=True
)

fig.show()


In [45]:
# 2. How do order statuses vary across the business?

fig = px.pie(names=df['order_status'].value_counts().index,
             values=df['order_status'].value_counts().values,
             title='Order Status Distribution')

fig.show()

In [46]:
# 3. monthly sales trend
data_of_month = df.groupby('purchase_month', as_index=False)['payment_value'].sum()

fig = px.line(data_frame=data_of_month,
                 x= 'purchase_month',
                 y= 'payment_value',
                 markers=True,
                 title="Monthly Sales Trend")

fig.show()

In [47]:
# 4. which cities place the most orders?

city_counts = df.groupby('customer_city', as_index=False)['order_id'].count()
city_counts.columns = ['customer_city', 'order_count']
city_counts = city_counts.sort_values(by='order_count', ascending=False)

top_cities = city_counts.head(10)

fig = px.bar(
    top_cities,
    x = 'customer_city',
    y = 'order_count',
    title='Top 10 Cities Placing the Most Orders',
    labels={'customer_city': 'City', 'order_count': 'Number of Orders'},
    text_auto=True,   # Displays the count on top of each bar
    color='order_count'
)

fig.show()

In [48]:
# 5. Which payment methods are most used?
fig = px.bar(
            x = df['payment_type'].value_counts().index,
            y = df['payment_type'].value_counts().values, 
            text_auto=True,
            title='payment methods are most used',
            color= df['payment_type'].value_counts().values)

fig.show()

In [49]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_value', 'payment_installments',
       'payment_type', 'price', 'freight_value', 'total_items', 'review_score',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state', 'is_delivered', 'has_review',
       'delivery_days', 'approval_hours', 'shipping_days', 'is_late',
       'purchase_year', 'purchase_month', 'purchase_day', 'purchase_hour',
       'purchase_day_name'],
      dtype='str')

In [50]:
# 6. Hourly Purchase Pattern Analysis

data = df.groupby('purchase_hour')['payment_value'].sum().reset_index()

fig = px.line(data,
    x = 'purchase_hour',
    y = 'payment_value',
    title= 'Hourly Purchase Pattern Analysis')

fig.show()

In [51]:
# 7. delivery delay affect review scores?

plot_df = df.dropna(subset=['review_score', 'delivery_days', 'is_late']).copy()


plot_df['review_score_cat'] = plot_df['review_score'].astype(str)


fig = px.box(
    plot_df,
    x = 'review_score_cat',
    y = 'delivery_days',
    color = 'is_late',
    title = 'Impact of Delivery Delays on Review Scores',
    labels = {
        'review_score_cat': 'Review Score (Stars)',
        'delivery_days': 'Actual Delivery Time (Days)',
        'is_late': 'Was Delivery Late?'
    },
    category_orders = {'review_score_cat': ['1', '2', '3', '4', '5']}
)


fig.update_layout(boxmode='group')
fig.show()


In [52]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_value', 'payment_installments',
       'payment_type', 'price', 'freight_value', 'total_items', 'review_score',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state', 'is_delivered', 'has_review',
       'delivery_days', 'approval_hours', 'shipping_days', 'is_late',
       'purchase_year', 'purchase_month', 'purchase_day', 'purchase_hour',
       'purchase_day_name'],
      dtype='str')

In [53]:
df.to_csv("clean_data.csv", index=False)